In [5]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df = \
    read_model_runs('../../data/processed/model-runs')



In [6]:
from read_model_runs import filter_complete_questions

models = ['gemma-3-27b-it', 'gemma-3-12b-it', 'gemma-3-4b-it']
firacs = ['FILA_', 'FILA_', 'FIR__', 'FI___', 'FIL__', '_____', 'unstructured']
question_long_df, question_wide_df, model_order, firac_order = filter_complete_questions(question_long_df, models, firacs)

print('shape:', question_long_df.shape)
print('# unique questions:', question_long_df['question_id'].nunique())
print("model order:", model_order)
print("firac order:", firac_order)


shape: (48978, 32)
# unique questions: 2721
model order: ['gemma-3-4b-it', 'gemma-3-12b-it', 'gemma-3-27b-it']
firac order: ['_____', 'unstructured', 'FIL__', 'FI___', 'FIR__', 'FILA_']


In [7]:
def get_api_keys():
    with open("openai_api_key.txt", "r") as f:
        api_keys = [
            line.strip()
            for line in f
            if line.strip() and not line.lstrip().startswith("#")
        ]
    return api_keys


api_key = get_api_keys()[0]

In [8]:
import json
import re

def normalize_json_like_string(s):
    # Primeiro: trocar aspas simples nas chaves por aspas duplas
    s = re.sub(r"'(\w+)'\s*:", r'"\1":', s)
    # Segundo: trocar aspas simples nos valores por aspas duplas,
    # mas sem mexer nas aspas que já são duplas
    s = re.sub(r":\s*'([^']*)'", r': "\1"', s)
    # Terceiro: trocar aspas simples dentro de listas ou dicionários
    s = re.sub(r"\[\s*'([^']*)'\s*\]", r'["\1"]', s)
    return s

import json

def remove_empty_values(data):
    """
    Remove todas as chaves cujo valor é vazio:
    "", "   ", None, [], {}
    Funciona recursivamente para dicionários aninhados.
    """
    if isinstance(data, dict):
        new_data = {}
        for key, value in data.items():
            # Limpar strings com espaços
            if isinstance(value, str) and value.strip() == "":
                continue
            # Ignorar valores vazios
            if value in (None, [], {}):
                continue
            # Recursão para estruturas internas
            cleaned_value = remove_empty_values(value)
            # Só mantém se não ficou vazio
            if cleaned_value not in ("", None, [], {}):
                new_data[key] = cleaned_value
        return new_data

    elif isinstance(data, list):
        # Limpa elementos vazios dentro de listas
        cleaned_list = [
            remove_empty_values(item)
            for item in data
            if item not in ("", None, [], {})
        ]
        return cleaned_list

    return data

def imprimir_leis(leis):
    leis_str = ""
    for chave, lista in leis.items():
        for item in lista:
            numero = item.get("numero", "")
            conteudo = item.get("conteudo", "")
            leis_str +=  f"{chave} -  {numero} : {conteudo}" + "; "
    return leis_str


def get_leis(rules, conteudo):
    rules = rules.replace("\"b\"", "b").replace("\'nı", "ni")
    rules = normalize_json_like_string(rules)

    if conteudo == "numeros":

        # função recursiva para extrair valores de "lei"
        def extract_lei(o):
            if isinstance(o, dict):
                current = [o["numero"]] if "numero" in o else []
                children = sum((extract_lei(v) for v in o.values()), [])
                return current + children
            elif isinstance(o, list):
                return sum((extract_lei(v) for v in o), [])
            else:
                return []

        # extrai os valores
        leis = extract_lei(json.loads(rules))

        # filtra arrays vazios (não gerará entradas vazias)
        leis_filtradas = [l for l in leis if l]

        return ", ".join(map(str, leis_filtradas))
    else:
        return imprimir_leis(remove_empty_values(json.loads(rules)))

leis = """ {'constituicao_federal': [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [], "codigo_tributario_nacional": [{"numero": "Art. 132 do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A pessoa jur\u00eddica de direito privado que resultar de fus\u00e3o, transforma\u00e7\u00e3o ou incorpora\u00e7\u00e3o de outra ou em outra \u00e9 respons\u00e1vel pelos tributos devidos at\u00e9 \u00e0 data do ato pelas pessoas jur\u00eddicas de direito privado fusionadas, transformadas ou incorporadas."}, {"numero": "Art. 133, inciso I do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 integral e isolada, no caso de o alienante cessar a explora\u00e7\u00e3o da atividade."}, {"numero": "Art. 133, inciso II do C\u00f3digo Tribut\u00e1rio Nacional", "conteudo": "A responsabilidade do adquirente pelos tributos devidos pelo alienante at\u00e9 a data da aquisi\u00e7\u00e3o do estabelecimento empresarial \u00e9 subsidi\u00e1ria, se o alienante prosseguir na explora\u00e7\u00e3o da atividade ou, dentro de seis meses contados da data da aliena\u00e7\u00e3o, iniciar nova atividade no mesmo ou em outro ramo de com\u00e9rcio, ind\u00fastria ou profiss\u00e3o."}], "principio_geral_do_direito": [], "outros": []}"""
leis_obtidas = get_leis(leis, "numeros")
assert leis_obtidas == 'Art. 132 do Código Tributário Nacional, Art. 133, inciso I do Código Tributário Nacional, Art. 133, inciso II do Código Tributário Nacional'


leis_obtidas = get_leis(leis, "conteudo")
print(leis_obtidas)

codigo_tributario_nacional -  Art. 132 do Código Tributário Nacional : A pessoa jurídica de direito privado que resultar de fusão, transformação ou incorporação de outra ou em outra é responsável pelos tributos devidos até à data do ato pelas pessoas jurídicas de direito privado fusionadas, transformadas ou incorporadas.; codigo_tributario_nacional -  Art. 133, inciso I do Código Tributário Nacional : A responsabilidade do adquirente pelos tributos devidos pelo alienante até a data da aquisição do estabelecimento empresarial é integral e isolada, no caso de o alienante cessar a exploração da atividade.; codigo_tributario_nacional -  Art. 133, inciso II do Código Tributário Nacional : A responsabilidade do adquirente pelos tributos devidos pelo alienante até a data da aquisição do estabelecimento empresarial é subsidiária, se o alienante prosseguir na exploração da atividade ou, dentro de seis meses contados da data da alienação, iniciar nova atividade no mesmo ou em outro ramo de comér

In [9]:
import pandas as pd

exam_df = pd.read_csv("../../data/processed/oab_with_firac_portuguese_shuffle.csv")

exam_df.head()

,question_id,pdf_filename,materia,oab_test_id,oab_question_id,language,enunciado,A,justificativa_A,B,...,justificativa_D,Facts,Issue,Rule,Application,Conclusion,correct_option,fact_count,rule_count,tema
0,oab-1.pdf-001,oab-1.pdf,ÉTICA PROFISSIONAL,XV,1,Portuguese,Abelardo é magistrado vinculado ao Tribunal de...,A incompatibilidade permanece mesmo que ocorra...,"Alternativa correta. Conforme o art. 28, §1º, ...",A incompatibilidade com a advocacia persiste m...,...,Afirmação falsa. A incompatibilidade persiste ...,"[\n ""Abelardo é magistrado vinculado ao Tribu...",Um magistrado em licença temporária para trata...,"{""constituicao_federal"": [{""numero"": ""Art. 95,...",Abelardo é um magistrado que obteve licença pa...,A incompatibilidade para o exercício da advoca...,A,5,3,INCOMPATIBILIDADE E IMPEDIMENTO
1,oab-1.pdf-002,oab-1.pdf,ÉTICA PROFISSIONAL,XV,2,Portuguese,"Fred, jovem advogado, é contratado para presta...",pode opor-se e postular assessoria da OAB.,Afirmação falsa. Não há previsão legal para qu...,pode recusar-se a propor a ação diante do pare...,...,Afirmação falsa. Todo advogado possui liberdad...,"[\n ""Fred, um jovem advogado, foi contratado ...","Nos termos do Código de Ética da Advocacia, qu...","{""constituicao_federal"": [], ""codigo_penal"": [...",O advogado possui liberdade e independência no...,O advogado pode recusar-se a propor a ação dia...,B,7,3,DIREITOS DO ADVOGADO
2,oab-1.pdf-003,oab-1.pdf,ÉTICA PROFISSIONAL,XV,3,Portuguese,O advogado Caio atuava representando os intere...,"Tício não pode ajuizar tal ação porque, como C...",Afirmação falsa. Os honorários são devidos (ar...,Tício não pode ajuizar tal ação porque o advog...,...,Afirmação falsa. O momento de ingresso do advo...,"[\n ""Um advogado, Caio, representava um clien...",Pode o advogado que recebeu substabelecimento ...,"{""constituicao_federal"": [], ""codigo_penal"": [...",Os honorários advocatícios de êxito são devido...,O advogado que recebeu substabelecimento com r...,B,6,4,ATIVIDADE DE ADVOCACIA
3,oab-1.pdf-004,oab-1.pdf,ÉTICA PROFISSIONAL,XV,4,Portuguese,"Os advogados X de Souza, Y dos Santos e Z de A...","É possível manter o nome do sócio falecido, in...",Afirmação falsa. Para manter o nome do sócio f...,É absolutamente vedada a manutenção do nome do...,...,Afirmação falsa. O EAOAB não fixa qualquer pra...,"[\n ""Os advogados X de Souza, Y dos Santos e ...",É juridicamente permitido manter o nome de um ...,"{""constituicao_federal"": [], ""codigo_penal"": [...",A legislação aplicável permite a permanência d...,É possível manter o nome do sócio falecido na ...,C,4,1,SOCIEDADE DE ADVOGADOS
4,oab-1.pdf-005,oab-1.pdf,ÉTICA PROFISSIONAL,XIV,5,Portuguese,Matheus é estagiário vinculado ao escritório R...,A renúncia deve ser notificada ao cliente pelo...,Alternativa correta. A renúncia deve ser comun...,"A renúncia ao mandato, sem respeitar o prazo l...",...,Afrirmação falsa. A renúnca configura ato unil...,"[\n ""Matheus, um estagiário vinculado ao escr...",Quais são os requisitos legais e as implicaçõe...,"{""constituicao_federal"": [], ""codigo_penal"": [...",Considerando a decisão do escritório de renunc...,A renúncia do mandato por parte do escritório ...,A,3,2,ATIVIDADE DE ADVOCACIA


In [10]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

def nli_classify(premise: str, hypothesis: str, model="gpt-4o-mini"):
    """
    Classifica a relação lógica entre premissa e hipótese.
    Retorna: ENTAILMENT | NEUTRAL | CONTRADICTION
    """

    prompt = f"""
You are performing Natural Language Inference (NLI).

Premise:
"{premise}"

Hypothesis:
"{hypothesis}"

Classify the relationship between the premise and the hypothesis.
Respond with exactly one of the following labels:
ENTAILMENT
NEUTRAL
CONTRADICTION
"""

    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0.0
    )

    # Extrai o texto retornado
    label = response.output_text.strip().upper()

    assert label in {"ENTAILMENT", "CONTRADICTION", "NEUTRAL"}, \
    f"Label inválido: {label}"

    return label


# -----------------------------
# Exemplo de uso
# -----------------------------
premise = "O réu confessou a prática do crime durante o interrogatório."
hypothesis = "O réu admitiu ter cometido o crime."

result = nli_classify(premise, hypothesis)
print(result)


ENTAILMENT


In [11]:
def test_entailment():
    premise = "O réu confessou a prática do crime durante o interrogatório."
    hypothesis = "O réu admitiu ter cometido o crime."

    label = nli_classify(premise, hypothesis)
    assert label == "ENTAILMENT", f"Esperado ENTAILMENT, obtido {label}"

def test_contradiction_1():
    premise = "O contrato foi devidamente assinado pelas duas partes."
    hypothesis = "O contrato não foi assinado por nenhuma das partes."

    label = nli_classify(premise, hypothesis)
    assert label == "CONTRADICTION", f"Esperado CONTRADICTION, obtido {label}"

def test_contradiction_2():
    premise = "Todo o patrimônio de Mateus caberá ntegralmente a Alberto."
    hypothesis = "Todo o patrimônio de Mateus caberá ntegralmente a Maria."

    label = nli_classify(premise, hypothesis)
    assert label == "CONTRADICTION", f"Esperado CONTRADICTION, obtido {label}"


def test_neutral():
    premise = "O juiz analisou o pedido de tutela de urgência."
    hypothesis = "O pedido foi deferido pelo juiz."

    label = nli_classify(premise, hypothesis)
    assert label == "NEUTRAL", f"Esperado NEUTRAL, obtido {label}"


# --------------------------------
# Execução manual dos testes
# --------------------------------
if __name__ == "__main__":
    #test_entailment()
    #test_contradiction_1()
    #test_contradiction_2()
    #test_neutral()
    print("Todos os testes passaram.")


Todos os testes passaram.


In [12]:
def get_relation(assumption, hypothesis):
    if assumption == hypothesis:
        return "ENTAILMENT"
    else:
        return nli_classify(assumption, hypothesis)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import threading
import numpy as np

# --------------------------------------------------
# Configurações
# --------------------------------------------------
models = ['gemma-3-27b-it']
firacs = ["_____", "FI___", "FIR__", "FILA_"]
MAX_WORKERS = 4
OUTPUT_PATH = "../../data/processed/question_entailment.csv"

lock = threading.Lock()

# --------------------------------------------------
# Carregar resultados existentes (se houver)
# --------------------------------------------------
if os.path.exists(OUTPUT_PATH):
    existing_df = pd.read_csv(OUTPUT_PATH)
    processed_keys = set(
        zip(
            existing_df["question_id"],
            existing_df["firac"],
            existing_df["field"],
            existing_df["assumption_model"],
            existing_df["hypothesis_model"]
        )
    )
    print(f"✔ {len(processed_keys)} registros já processados — pulando")
else:
    processed_keys = set()

# --------------------------------------------------
# Função unitária (SEM tolerância a falhas)
# --------------------------------------------------
def run_relation(
    question_id,
    materia,
    tema,
    firac,
    field,
    assumption_model,
    hypothesis_model,
    is_correct,
    assumption,
    hypothesis
):
    
    if firac == "FIR__" and field == "rule":
        relation = "ENTAILMENT"
    elif firac == "FILA_" and field == "rule":
        relation = "ENTAILMENT"
    elif firac == "FIRA_" and field == "rule":
        relation = "ENTAILMENT"
    elif firac == "FILA_" and field == "application":
        relation = "ENTAILMENT"
    elif firac == "FIRA_" and field == "application":
        relation = "ENTAILMENT"
    else:
        relation = get_relation(assumption, hypothesis)  # ← se quebrar, tudo para

    return {
        "question_id": question_id,
        "materia": materia,
        "tema": tema,
        "firac": firac,
        "field": field,
        "assumption_model": assumption_model,
        "hypothesis_model": hypothesis_model,
        "is_correct": is_correct,
        "relation_label": relation,
        "assumption": assumption,
        "hypothesis": hypothesis,
        "created_at": datetime.now()
    }

# --------------------------------------------------
# Construção das tarefas
# --------------------------------------------------
tasks = []

question_long_df = question_long_df[
    (question_long_df['materia'].isin(['DIREITO DO TRABALHO', 'PROCESSO DO TRABALHO', 'DIREITO PENAL', 'PROCESSO PENAL', 'DIREITO CIVIL', 'PROCESSO CIVIL'])) 

]


unique_question_ids = np.random.permutation(question_long_df["question_id"].unique())
unique_question_ids = question_long_df["question_id"].unique()
print('unique question ids:', len(unique_question_ids))


print("Scheduling tasks...")
for question_id in unique_question_ids:
    for model in models:
        for firac in firacs:

            question_gt_row = exam_df[exam_df.question_id == question_id].iloc[0]
            materia = question_gt_row['materia']
            tema = question_gt_row['tema']

            question_filtered_df = question_long_df[
                (question_long_df["question_id"] == question_id) &
                (question_long_df["model_name"] == model) &
                (question_long_df["firac"] == firac)
            ]

            assert question_filtered_df.shape[0] == 1

            question_filtered_row = question_filtered_df.iloc[0]

            # Ground truth
            gt_rule = get_leis(question_gt_row["Rule"], "conteudo")
            gt_application = question_gt_row["Application"]
            gt_conclusion = question_gt_row["Conclusion"]

            # Modelo
            model_rule = question_filtered_row["Rule"]
            model_application = question_filtered_row["Application"]
            model_conclusion = question_filtered_row["Conclusion"]

            candidates = [
                ("rule", gt_rule, model_rule),
                ("application", gt_application, model_application),
                ("conclusion", gt_conclusion, model_conclusion),
            ]

            assumption_model="ground-truth"
            for field, assumption, hypothesis in candidates:
                key = (question_id, firac, field, "ground-truth", model)
                if key in processed_keys:
                    continue

                tasks.append(dict(
                    question_id=question_id,
                    materia=materia,
                    tema=tema,
                    firac=firac,
                    field=field,
                    assumption_model=assumption_model,
                    hypothesis_model=model,
                    is_correct=question_filtered_row["is_correct"],
                    assumption=assumption,
                    hypothesis=hypothesis,
                ))

print(f"🚀 {len(tasks)} tarefas novas a executar")

header_written = os.path.exists(OUTPUT_PATH)

print("Triggering tasks...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(run_relation, **task) for task in tasks]

    try:
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Chamadas get_relation"
        ):
            result = future.result()  # ← exceção explode aqui

            with lock:
                pd.DataFrame([result]).to_csv(
                    OUTPUT_PATH,
                    mode="a",
                    index=False,
                    header=not header_written,
                    encoding="utf-8-sig"
                )
                header_written = True

    except Exception:
        # Cancela tudo imediatamente
        for f in futures:
            f.cancel()

        executor.shutdown(wait=False, cancel_futures=True)
        raise  # ← propaga erro original (stack trace limpo)


✔ 14838 registros já processados — pulando
unique question ids: 1128
Scheduling tasks...
🚀 8731 tarefas novas a executar
Triggering tasks...


Chamadas get_relation:  30%|██▉       | 2578/8731 [07:15<17:19,  5.92it/s]  


In [1]:
import pandas as pd

question_entailment_long_df = pd.read_csv("../../data/processed/question_entailment.csv")

print(question_entailment_long_df.shape)

question_entailment_long_df

(17416, 12)


,question_id,materia,tema,firac,field,assumption_model,hypothesis_model,is_correct,relation_label,assumption,hypothesis,created_at
0,oab-153.pdf-007,DIREITO CIVIL,DIREITO DAS OBRIGAÇÕES,_____,application,ground-truth,gemma-3-27b-it,True,ENTAILMENT,Ao analisar a responsabilidade em caso de fale...,"A alternativa A está incorreta, pois as exceçõ...",2025-12-25 06:09:09.867290
1,oab-153.pdf-007,DIREITO CIVIL,DIREITO DAS OBRIGAÇÕES,_____,rule,ground-truth,gemma-3-27b-it,True,NEUTRAL,codigo_civil - Art. 276 do Código Civil : Em ...,"O Código Civil, nos artigos 275 a 281, discipl...",2025-12-25 06:09:10.598696
2,oab-153.pdf-007,DIREITO CIVIL,DIREITO DAS OBRIGAÇÕES,_____,conclusion,ground-truth,gemma-3-27b-it,True,ENTAILMENT,"Em um regime de solidariedade passiva, os herd...","A alternativa C está correta, pois a perda do ...",2025-12-25 06:09:10.714158
3,oab-153.pdf-007,DIREITO CIVIL,DIREITO DAS OBRIGAÇÕES,FI___,application,ground-truth,gemma-3-27b-it,True,ENTAILMENT,Ao analisar a responsabilidade em caso de fale...,"Na solidariedade passiva, as exceções são opon...",2025-12-25 06:09:11.432044
4,oab-153.pdf-007,DIREITO CIVIL,DIREITO DAS OBRIGAÇÕES,FI___,rule,ground-truth,gemma-3-27b-it,True,NEUTRAL,codigo_civil - Art. 276 do Código Civil : Em ...,"O Código Civil, nos artigos 835 a 842, discipl...",2025-12-25 06:09:11.732750
...,...,...,...,...,...,...,...,...,...,...,...,...
17411,oab-342.pdf-207,PROCESSO DO TRABALHO,EXECUÇÃO TRABALHISTA,FI___,rule,ground-truth,gemma-3-27b-it,True,ENTAILMENT,"clt - Art. 879, §2º da CLT : A parte que não ...",O art. 832 da CLT estabelece que nos embargos ...,2025-12-26 05:42:36.014157
17412,oab-342.pdf-207,PROCESSO DO TRABALHO,EXECUÇÃO TRABALHISTA,FI___,conclusion,ground-truth,gemma-3-27b-it,True,ENTAILMENT,Está preclusa a arguição de matérias que impug...,A preclusão da ré em impugnar os cálculos de l...,2025-12-26 05:42:36.402573
17413,oab-342.pdf-207,PROCESSO DO TRABALHO,EXECUÇÃO TRABALHISTA,FILA_,rule,ground-truth,gemma-3-27b-it,True,ENTAILMENT,"clt - Art. 879, §2º da CLT : A parte que não ...","Art. 879, §2º da CLT",2025-12-26 05:42:36.402573
17414,oab-342.pdf-207,PROCESSO DO TRABALHO,EXECUÇÃO TRABALHISTA,FILA_,application,ground-truth,gemma-3-27b-it,True,ENTAILMENT,A parte ré silenciou-se e não apresentou impug...,A parte ré silenciou-se e não apresentou impug...,2025-12-26 05:42:36.419755


In [2]:
import pandas as pd

# --------------------------------------------------
# 0) Cópia defensiva
# --------------------------------------------------
df = question_entailment_long_df.copy()

# --------------------------------------------------
# 1) Mapear relation_label para valores numéricos
# --------------------------------------------------
relation_map = {
    "CONTRADICTION": -1,
    "NEUTRAL": 0,
    "ENTAILMENT": 1
}

df["relation_value"] = df["relation_label"].map(relation_map)

# --------------------------------------------------
# 2) Criar nome da coluna firac + field
# --------------------------------------------------
df["firac_field"] = df["firac"].astype(str) + "|" + df["field"].astype(str)


# --------------------------------------------------
# 3) Pivotar para formato wide
# --------------------------------------------------
question_entailment_wide_df = (
    df
    .pivot_table(
        index=["question_id"],
        columns="firac_field",
        values="relation_value",
        aggfunc="first"   # assume 1 valor por combinação
    )
    .reset_index()
)

# --------------------------------------------------
# Resultado final
# --------------------------------------------------
question_entailment_wide_df = question_entailment_wide_df.dropna()
question_entailment_wide_df.to_csv("../../data/processed/question_entailment_wide.csv", index=False)

print(question_entailment_wide_df.shape)

question_entailment_wide_df

(1447, 13)


firac_field,question_id,FILA_|application,FILA_|conclusion,FILA_|rule,FIR__|application,FIR__|conclusion,FIR__|rule,FI___|application,FI___|conclusion,FI___|rule,_____|application,_____|conclusion,_____|rule
0,oab-1.pdf-002,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
1,oab-1.pdf-003,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,-1.0,-1.0,0.0
2,oab-1.pdf-004,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,oab-1.pdf-005,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,1.0
4,oab-1.pdf-006,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,oab-96.pdf-079,1.0,0.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0
1450,oab-97.pdf-008,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
1451,oab-97.pdf-012,1.0,0.0,1.0,1.0,0.0,1.0,-1.0,0.0,0.0,-1.0,-1.0,0.0
1452,oab-98.pdf-018,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0


In [3]:
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 0) Ordem desejada para field
# --------------------------------------------------
field_order = ["rule", "application", "conclusion"]

df = question_entailment_long_df.copy()
df["field"] = pd.Categorical(
    df["field"],
    categories=field_order,
    ordered=True
)

# --------------------------------------------------
# 1) Contagem por field + firac + relation_label
# --------------------------------------------------
counts_df = (
    df
    .groupby(["field", "firac", "relation_label"])
    .size()
    .reset_index(name="n")
)

# --------------------------------------------------
# 2) Pivot para formato largo
# --------------------------------------------------
pivot_df = (
    counts_df
    .pivot_table(
        index=["field", "firac"],
        columns="relation_label",
        values="n",
        fill_value=0
    )
)

# --------------------------------------------------
# 3) Normalização por linha (percentual)
# --------------------------------------------------
pct_df = (
    pivot_df
    .div(pivot_df.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
    .sort_index(level="field")
)

# --------------------------------------------------
# 4) Estilização ACL-style
# --------------------------------------------------
styled_pct_df = (
    pct_df.style
    .format("{:.1f}%")
    .background_gradient(
        cmap="RdYlGn",
        axis=1
    )
    .set_caption(
        "Distribution of NLI Labels (%) by Field and FIRAC Level"
    )
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("margin-bottom", "8px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center")
            ]
        }
    ])
)

# --------------------------------------------------
# 5) Mostrar tabela bonita
# --------------------------------------------------
display(styled_pct_df)


C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\92425768.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["field", "firac", "relation_label"])
C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\92425768.py:31: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


In [4]:
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 0) Ordem desejada para field
# --------------------------------------------------
field_order = ["rule", "application", "conclusion"]

df = question_entailment_long_df.copy()
df["field"] = pd.Categorical(
    df["field"],
    categories=field_order,
    ordered=True
)

# --------------------------------------------------
# Função para gerar matriz percentual
# --------------------------------------------------
def build_pct_matrix(df_subset):
    counts_df = (
        df_subset
        .groupby(["field", "firac", "relation_label"])
        .size()
        .reset_index(name="n")
    )

    pivot_df = (
        counts_df
        .pivot_table(
            index=["field", "firac"],
            columns="relation_label",
            values="n",
            fill_value=0
        )
    )

    pct_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100
    pct_df = pct_df.round(1)
    pct_df = pct_df.sort_index(level="field")

    return pct_df


# --------------------------------------------------
# Função de estilização bonita (ACL-style)
# --------------------------------------------------
def style_pct_table(df, title):
    styled = (
        df.style
        .format("{:.1f}%")
        .background_gradient(
            cmap="RdYlGn",
            axis=1
        )
        .set_caption(title)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("margin-bottom", "8px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("font-weight", "bold"),
                    ("text-align", "center")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center")
                ]
            }
        ])
    )
    return styled


# --------------------------------------------------
# 1) Tabela — is_correct == True
# --------------------------------------------------
pct_correct_true = build_pct_matrix(df[df["is_correct"] == True])

display(
    style_pct_table(
        pct_correct_true,
        "Distribution of NLI Labels (%) — Correct Answers (is_correct = True)"
    )
)

# --------------------------------------------------
# 2) Tabela — is_correct == False
# --------------------------------------------------
pct_correct_false = build_pct_matrix(df[df["is_correct"] == False])

display(
    style_pct_table(
        pct_correct_false,
        "Distribution of NLI Labels (%) — Incorrect Answers (is_correct = False)"
    )
)


C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\2630471667.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["field", "firac", "relation_label"])
C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\2630471667.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\2630471667.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["field", "firac", "relation_label"])
C:\Users\pedro\AppData\Local\Temp\ipykernel_12052\2630471667.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


In [5]:
import pandas as pd

# --------------------------------------------------
# 0) Cópia defensiva
# --------------------------------------------------
df = question_entailment_long_df.copy()

# Esperado no dataframe:
# - materia
# - field
# - relation_label

# --------------------------------------------------
# 1) Criar flag binária de CONTRADICTION
# --------------------------------------------------
df["is_contradiction"] = (df["relation_label"] == "CONTRADICTION").astype(int)

# --------------------------------------------------
# 2) Agregar por materia + field
# --------------------------------------------------
agg_df = (
    df
    .groupby(["materia", "field"])
    .agg(
        n_total=("relation_label", "size"),
        n_contradiction=("is_contradiction", "sum")
    )
    .reset_index()
)

# --------------------------------------------------
# 3) Calcular taxa percentual de CONTRADICTION
# --------------------------------------------------
agg_df["pct_contradiction"] = (
    agg_df["n_contradiction"] / agg_df["n_total"] * 100
)

# --------------------------------------------------
# 4) Pivotar: linhas = materia, colunas = field
# --------------------------------------------------
contradiction_rate_df = (
    agg_df
    .pivot_table(
        index="materia",
        columns="field",
        values="pct_contradiction"
    )
)

# --------------------------------------------------
# 5) Garantir ordem FIRAC fixa nas colunas
# --------------------------------------------------
firac_order = ["rule", "application", "conclusion"]
firac_cols = [c for c in firac_order if c in contradiction_rate_df.columns]
contradiction_rate_df = contradiction_rate_df[firac_cols]

# --------------------------------------------------
# 6) Adicionar coluna count (total de observações por matéria)
# --------------------------------------------------
count_df = (
    agg_df
    .groupby("materia")["n_total"]
    .sum()
    .rename("count")
)

contradiction_rate_df = contradiction_rate_df.join(count_df)

# --------------------------------------------------
# 7) Calcular média de CONTRADICTION por matéria
# --------------------------------------------------
contradiction_rate_df["mean_contradiction"] = (
    contradiction_rate_df[firac_cols].mean(axis=1)
)

# --------------------------------------------------
# 8) Ordenar linhas por mean_contradiction
# --------------------------------------------------
contradiction_rate_df = (
    contradiction_rate_df
    .sort_values("mean_contradiction", ascending=False)
)

# --------------------------------------------------
# 9) Arredondar para leitura
# --------------------------------------------------
contradiction_rate_df = contradiction_rate_df.round(1)

# --------------------------------------------------
# Resultado final
# --------------------------------------------------
contradiction_rate_df.head(10000)


,rule,application,conclusion,count,mean_contradiction
materia,,,,,
DIREITO INTERNACIONAL,5.0,40.0,30.0,120,25.0
DIREITO PREVIDENCIÁRIO,10.0,25.0,30.0,60,21.7
DIREITO ELEITORAL,0.0,37.5,25.0,24,20.8
DIREITO DO TRABALHO,10.4,25.5,25.5,2256,20.5
DIREITO PENAL,5.3,25.0,29.0,396,19.8
DIREITO TRIBUTÁRIO,6.1,21.3,22.4,1524,16.6
PROCESSO DO TRABALHO,9.2,21.6,18.8,1890,16.5
PROCESSO CIVIL,6.7,25.0,17.3,312,16.3
DIREITO ADMINISTRATIVO,7.1,21.4,16.4,420,15.0


In [ ]:
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 0) Ordem desejada para field
# --------------------------------------------------
field_order = ["rule", "application", "conclusion"]

df = question_entailment_long_df.copy()
df["field"] = pd.Categorical(
    df["field"],
    categories=field_order,
    ordered=True
)

# --------------------------------------------------
# Função para gerar matriz percentual
# --------------------------------------------------
def build_pct_matrix(df_subset):
    counts_df = (
        df_subset
        .groupby(["field", "firac", "relation_label"])
        .size()
        .reset_index(name="n")
    )

    pivot_df = (
        counts_df
        .pivot_table(
            index=["field", "firac"],
            columns="relation_label",
            values="n",
            fill_value=0
        )
    )

    pct_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100
    pct_df = pct_df.round(1)
    pct_df = pct_df.sort_index(level="field")

    return pct_df


# --------------------------------------------------
# Função de estilização bonita (ACL-style)
# --------------------------------------------------
def style_pct_table(df, title):
    styled = (
        df.style
        .format("{:.1f}%")
        .background_gradient(
            cmap="RdYlGn",
            axis=1
        )
        .set_caption(title)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("margin-bottom", "8px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("font-weight", "bold"),
                    ("text-align", "center")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center")
                ]
            }
        ])
    )
    return styled


# --------------------------------------------------
# 1) Tabela — is_correct == True
# --------------------------------------------------
pct_correct_true = build_pct_matrix(df[df["is_correct"] == True])

display(
    style_pct_table(
        pct_correct_true,
        "Distribution of NLI Labels (%) — Correct Answers (is_correct = True)"
    )
)

# --------------------------------------------------
# 2) Tabela — is_correct == False
# --------------------------------------------------
pct_correct_false = build_pct_matrix(df[df["is_correct"] == False])

display(
    style_pct_table(
        pct_correct_false,
        "Distribution of NLI Labels (%) — Incorrect Answers (is_correct = False)"
    )
)
